In [9]:
import sqlite3
import pandas as pd
import plotly.express as px

In [10]:
conn = sqlite3.connect("airspace.db")

query = """
SELECT 
    f.flight_id,
    f.callsign,
    f.airline_code,
    a.origin_country,
    t.latitude,
    t.longitude,
    ROUND(t.baro_alt * 3.28084) AS altitude_ft,
    ROUND(t.velocity_knots, 1) AS velocity_knots,
    ROUND(t.vertical_rate_fpm, 1) AS vertical_rate_fpm,
    CASE 
        WHEN t.vertical_rate_fpm > 500 THEN 'Climb'
        WHEN t.vertical_rate_fpm < -500 THEN 'Descent'
        ELSE 'Cruise'
    END AS flight_phase,
    CASE 
        WHEN ABS(t.vertical_rate_fpm) >= 2500 THEN 'Anomaly'
        ELSE 'Normal'
    END AS operation_status
FROM flights f
JOIN telemetry_logs t ON f.flight_id = t.flight_id
JOIN aircraft a ON f.icao24 = a.icao24
WHERE t.latitude IS NOT NULL 
  AND t.longitude IS NOT NULL;
"""

df = pd.read_sql_query(query, conn)
conn.close()
df.head()

,flight_id,callsign,airline_code,origin_country,latitude,longitude,altitude_ft,velocity_knots,vertical_rate_fpm,flight_phase,operation_status
0,1,PGT582,PGT,Turkey,39.4801,29.9490,33000.0,444.2,0.0,Cruise,Normal
1,2,PGT4260,PGT,Turkey,38.7866,30.1732,30925.0,415.2,-895.7,Descent,Normal
2,3,PGT70AF,PGT,Turkey,40.6136,30.7953,26225.0,455.5,1472.4,Climb,Normal
3,4,WZZ1199,WZZ,Hungary,42.0975,43.7132,20000.0,321.7,0.0,Cruise,Normal
4,5,MGH849,MGH,Turkey,37.8023,30.9479,19650.0,431.4,2559.1,Climb,Anomaly


In [11]:
fig_map = px.scatter_geo(
    df,
    lat="latitude",
    lon="longitude",
    color="flight_phase",
    color_discrete_map={"Climb": "#2ecc71", "Cruise": "#3498db", "Descent": "#e74c3c"},
    symbol="operation_status",
    hover_name="callsign",
    hover_data={
        "airline_code": True,
        "altitude_ft": ":,d",
        "velocity_knots": ":.1f",
        "vertical_rate_fpm": True,
        "latitude": False,
        "longitude": False
    },
    labels={
        "flight_phase":"Flight Phase",
        "operation_status":"Operation Status",
        "callsign":"Callsign"
    },
    fitbounds="locations",
    title="Live Air Traffic Overview",
    height=600,
)

fig_map.update_geos(
    resolution=50,
    showcountries=True,
    showcoastlines=True,
    showland=True,
    countrycolor="#2c3e50",
    coastlinecolor="#7f8c8d",
    landcolor="#f8f9fa"
)

fig_map.show()

In [12]:
fleet = df[df["airline_code"].notna()].groupby("airline_code")["flight_id"].nunique().reset_index(name="total_flights").sort_values(by="total_flights", ascending=False).head(10)

fig = px.bar(
    fleet, 
    y="total_flights", 
    x="airline_code",
    title="Top 10 Airlines by Flight Volume",
    labels={"airline_code":"Airline Code", "total_flights":"Active Flight Count"},
    color="total_flights",
    text_auto=True,
    color_continuous_scale="Blues"
)

fig.update_traces(textposition="outside")
fig.update_layout(yaxis=dict(showgrid=True), height=450)

fig.show()

In [13]:
fig = px.scatter(
    df,
    x="velocity_knots",
    y="altitude_ft",
    color="operation_status",
    symbol="flight_phase",
    color_discrete_map={"Normal": "#34495e", "Anomaly": "#e74c3c"},
    labels={"velocity_knots":"Ground Speed (knots)", "altitude_ft":"Barometric Altitude (ft)", "operation_status":"Operational Status", "flight_phase":"Flight Phase"},
    title="Altitude vs. Speed Distribution (Anomaly Detection)",
    height=500,
    hover_name="callsign",
    hover_data=["airline_code", "vertical_rate_fpm"]
)

fig.update_layout(template="plotly_white")

fig.show()

In [ ]:
fix = px.box(
    df,
    x="flight_phase",
    y="velocity_knots",
    color="flight_phase",
    points="all",
    title="Speed Distribution Across Flight Phases",
    labels={
        "flight_phase": "Flight Phase",
        "velocity_knots": "Ground Speed (knots)"},
    
)

fix.show()

In [ ]:
fig = px.histogram(
    df,
    x="altitude_ft",
    color="flight_phase",
    title="Altitude Distribution by Flight Phase",
    labels={
        "altitude_ft": "Altitude (ft)",
        "flight_phase": "Flight Phase",
        "count": "Flight Count"},
)


fig.show()